# Native Quditkit Circuit Optimizer — Showcase

Demonstrates every optimization rule implemented in `my_genQC.platform.backends.circuit_optimizer`.

The optimizer works natively on quditkit `QuantumCircuit.ops` without any qiskit round-trip.
It runs two passes in a fixed-point loop until no further reduction is possible:

1. **Self-inverse cancellation** — using commutation to look past intervening gates
2. **Template rewriting** — replacing known 3- and 5-gate patterns with shorter equivalents

Gate set in scope: `H`, `X`, `Z`, `CNOT`, `SWAP`

> `CCX` (Toffoli) is not a Clifford gate and is not supported by quditkit — it never appears in quditkit circuits.

Convention: `op[1] = (control, target)` for two-qubit gates.

In [1]:
import os
os.chdir('../..')

import sys
import numpy as np
from qudit_sim.circuit import QuantumCircuit
from qudit_sim.predefined_gates import H_gate, X_gate, Z_gate, CNOT_gate, SWAP_gate

from src.my_genQC.platform.backends.circuit_optimizer import optimize_ops

In [2]:
def fmt_ops(ops):
    """Return a readable string like 'H(0) · CNOT(0,1) · H(0)'."""
    if not ops:
        return "∅  (identity)"
    parts = []
    for g, q, _ in ops:
        parts.append(f"{g.name}({','.join(str(x) for x in q)})")
    return " · ".join(parts)


def get_unitary(ops, n):
    if not ops:
        return np.eye(2**n, dtype=complex)
    qc = QuantumCircuit(num_qudits=n, dim=2)
    for g, q, d in ops:
        qc.append(g, q, dagger=d)
    return qc.get_unitary()


def show(label, ops, n, expect_len=None):
    """Optimize ops, print before/after, and assert the unitary is preserved."""
    opt = optimize_ops(list(ops))
    U_before = get_unitary(ops, n)
    U_after  = get_unitary(opt, n)
    ok = np.allclose(U_before, U_after, atol=1e-10)
    tag = "✓" if ok else "✗ UNITARY MISMATCH"
    print(f"[{label}]")
    print(f"  before ({len(ops):2d}): {fmt_ops(ops)}")
    print(f"  after  ({len(opt):2d}): {fmt_ops(opt)}  {tag}")
    if expect_len is not None:
        assert len(opt) == expect_len, f"expected {expect_len} gates, got {len(opt)}"
    print()

---
## 1. Self-Inverse Cancellation

Every gate `G` in the set `{H, X, Z, CNOT, CCX, SWAP}` satisfies `G·G = I`.  
Two identical applications on the same qubits always cancel out.

In [3]:
show("H · H = I",           [(H_gate,    (0,),    False), (H_gate,    (0,),    False)], n=1, expect_len=0)
show("X · X = I",           [(X_gate,    (0,),    False), (X_gate,    (0,),    False)], n=1, expect_len=0)
show("Z · Z = I",           [(Z_gate,    (0,),    False), (Z_gate,    (0,),    False)], n=1, expect_len=0)
show("CNOT · CNOT = I",     [(CNOT_gate, (0, 1),  False), (CNOT_gate, (0, 1),  False)], n=2, expect_len=0)
show("SWAP · SWAP = I",     [(SWAP_gate, (0, 1),  False), (SWAP_gate, (0, 1),  False)], n=2, expect_len=0)
show("SWAP(1,0)·SWAP(0,1)", [(SWAP_gate, (1, 0),  False), (SWAP_gate, (0, 1),  False)], n=2, expect_len=0)

[H · H = I]
  before ( 2): H(0) · H(0)
  after  ( 0): ∅  (identity)  ✓

[X · X = I]
  before ( 2): X(0) · X(0)
  after  ( 0): ∅  (identity)  ✓

[Z · Z = I]
  before ( 2): Z(0) · Z(0)
  after  ( 0): ∅  (identity)  ✓

[CNOT · CNOT = I]
  before ( 2): CNOT(0,1) · CNOT(0,1)
  after  ( 0): ∅  (identity)  ✓

[SWAP · SWAP = I]
  before ( 2): SWAP(0,1) · SWAP(0,1)
  after  ( 0): ∅  (identity)  ✓

[SWAP(1,0)·SWAP(0,1)]
  before ( 2): SWAP(1,0) · SWAP(0,1)
  after  ( 0): ∅  (identity)  ✓



**SWAP order normalisation:** `SWAP(1,0)` and `SWAP(0,1)` describe the same physical operation,
so the optimizer normalises qubit order to `(min, max)` before matching.

---
## 2. Commutation-Enabled Cancellation

A self-inverse gate `G` does not need to be *adjacent* to its pair — the optimizer scans
forward and skips gates that **commute** with `G`.  Scanning stops as soon as a blocking
gate is encountered.

| Gates | Commute? | Condition |
|-------|----------|-----------|
| any two | ✅ | No shared qubits |
| `Z(q)` and `CNOT(c,t)` | ✅ | `q == c` (Z on control) |
| `X(q)` and `CNOT(c,t)` | ✅ | `q == t` (X on target) |
| `CNOT(c,t1)` and `CNOT(c,t2)` | ✅ | same control, different targets |
| `CNOT(c1,t)` and `CNOT(c2,t)` | ✅ | different controls, same target |
| `H(q)` and any gate on `q` | ❌ | H always blocks |
| `Z(q)` and `CNOT(c,t)` | ❌ | `q == t` (Z on target) |
| `X(q)` and `CNOT(c,t)` | ❌ | `q == c` (X on control) |

In [4]:
# H(0) commutes past Z(1) (disjoint qubits) and cancels with the second H(0)
show(
    "H(0) · Z(1) · H(0)  →  Z(1)  [disjoint commutation]",
    [(H_gate, (0,), False), (Z_gate, (1,), False), (H_gate, (0,), False)],
    n=2, expect_len=1,
)

[H(0) · Z(1) · H(0)  →  Z(1)  [disjoint commutation]]
  before ( 3): H(0) · Z(1) · H(0)
  after  ( 1): Z(1)  ✓



In [5]:
# Z on the CNOT control commutes → the two CNOTs cancel, Z(0) remains
show(
    "CNOT(0,1) · Z(0) · CNOT(0,1)  →  Z(0)  [Z on control commutes]",
    [(CNOT_gate, (0,1), False), (Z_gate, (0,), False), (CNOT_gate, (0,1), False)],
    n=2, expect_len=1,
)

[CNOT(0,1) · Z(0) · CNOT(0,1)  →  Z(0)  [Z on control commutes]]
  before ( 3): CNOT(0,1) · Z(0) · CNOT(0,1)
  after  ( 1): Z(0)  ✓



In [6]:
# X on the CNOT target commutes → the two CNOTs cancel, X(1) remains
show(
    "CNOT(0,1) · X(1) · CNOT(0,1)  →  X(1)  [X on target commutes]",
    [(CNOT_gate, (0,1), False), (X_gate, (1,), False), (CNOT_gate, (0,1), False)],
    n=2, expect_len=1,
)

[CNOT(0,1) · X(1) · CNOT(0,1)  →  X(1)  [X on target commutes]]
  before ( 3): CNOT(0,1) · X(1) · CNOT(0,1)
  after  ( 1): X(1)  ✓



In [7]:
# CNOT(0,1) and CNOT(0,2) share only the control → they commute
# So CNOT(0,1) looks past CNOT(0,2) and cancels with the third gate
show(
    "CNOT(0,1) · CNOT(0,2) · CNOT(0,1)  →  CNOT(0,2)  [same-control commutation]",
    [(CNOT_gate, (0,1), False), (CNOT_gate, (0,2), False), (CNOT_gate, (0,1), False)],
    n=3, expect_len=1,
)

[CNOT(0,1) · CNOT(0,2) · CNOT(0,1)  →  CNOT(0,2)  [same-control commutation]]
  before ( 3): CNOT(0,1) · CNOT(0,2) · CNOT(0,1)
  after  ( 1): CNOT(0,2)  ✓



In [8]:
# CNOT(0,2) and CNOT(1,2) share only the target → they commute
show(
    "CNOT(0,2) · CNOT(1,2) · CNOT(0,2)  →  CNOT(1,2)  [same-target commutation]",
    [(CNOT_gate, (0,2), False), (CNOT_gate, (1,2), False), (CNOT_gate, (0,2), False)],
    n=3, expect_len=1,
)

[CNOT(0,2) · CNOT(1,2) · CNOT(0,2)  →  CNOT(1,2)  [same-target commutation]]
  before ( 3): CNOT(0,2) · CNOT(1,2) · CNOT(0,2)
  after  ( 1): CNOT(1,2)  ✓



In [9]:
print("--- blocking cases (nothing should cancel) ---\n")

# Z on target does NOT commute → cancellation is blocked
show(
    "CNOT(0,1) · Z(1) · CNOT(0,1)  →  unchanged  [Z on target blocks]",
    [(CNOT_gate, (0,1), False), (Z_gate, (1,), False), (CNOT_gate, (0,1), False)],
    n=2, expect_len=3,
)

# X on control does NOT commute → cancellation is blocked
show(
    "CNOT(0,1) · X(0) · CNOT(0,1)  →  unchanged  [X on control blocks]",
    [(CNOT_gate, (0,1), False), (X_gate, (0,), False), (CNOT_gate, (0,1), False)],
    n=2, expect_len=3,
)

# H on shared qubit always blocks
show(
    "CNOT(0,1) · H(0) · CNOT(0,1)  →  unchanged  [H always blocks]",
    [(CNOT_gate, (0,1), False), (H_gate, (0,), False), (CNOT_gate, (0,1), False)],
    n=2, expect_len=3,
)

--- blocking cases (nothing should cancel) ---

[CNOT(0,1) · Z(1) · CNOT(0,1)  →  unchanged  [Z on target blocks]]
  before ( 3): CNOT(0,1) · Z(1) · CNOT(0,1)
  after  ( 3): CNOT(0,1) · Z(1) · CNOT(0,1)  ✓

[CNOT(0,1) · X(0) · CNOT(0,1)  →  unchanged  [X on control blocks]]
  before ( 3): CNOT(0,1) · X(0) · CNOT(0,1)
  after  ( 3): CNOT(0,1) · X(0) · CNOT(0,1)  ✓

[CNOT(0,1) · H(0) · CNOT(0,1)  →  unchanged  [H always blocks]]
  before ( 3): CNOT(0,1) · H(0) · CNOT(0,1)
  after  ( 3): CNOT(0,1) · H(0) · CNOT(0,1)  ✓



---
## 3. Template Rewriting

The template pass scans the circuit left-to-right and replaces known gate patterns
with shorter equivalents.  All patterns are verified by the numerical unitary check below.

| Pattern | Result | Gate saving |
|---------|--------|-------------|
| `H · Z · H` (same qubit) | `X` | −2 |
| `H · X · H` (same qubit) | `Z` | −2 |
| `CNOT(a,b) · CNOT(b,a) · CNOT(a,b)` | `SWAP(a,b)` | −2 |
| `H(a) · H(b) · CNOT(a,b) · H(a) · H(b)` | `CNOT(b,a)` | −4 |

> **Note:** `H(a) · CNOT(a,b) · H(a) ≠ CNOT(b,a)` — the 3-gate version does **not** hold;
> conjugation by Hadamard requires H on **both** qubits.

In [10]:
# H · Z · H = X  (Hadamard conjugation)
show(
    "H(0) · Z(0) · H(0)  →  X(0)",
    [(H_gate, (0,), False), (Z_gate, (0,), False), (H_gate, (0,), False)],
    n=1, expect_len=1,
)

[H(0) · Z(0) · H(0)  →  X(0)]
  before ( 3): H(0) · Z(0) · H(0)
  after  ( 1): X(0)  ✓



In [11]:
# H · X · H = Z  (Hadamard conjugation)
show(
    "H(0) · X(0) · H(0)  →  Z(0)",
    [(H_gate, (0,), False), (X_gate, (0,), False), (H_gate, (0,), False)],
    n=1, expect_len=1,
)

[H(0) · X(0) · H(0)  →  Z(0)]
  before ( 3): H(0) · X(0) · H(0)
  after  ( 1): Z(0)  ✓



In [12]:
# CNOT(a,b) · CNOT(b,a) · CNOT(a,b) = SWAP(a,b)  (standard 3-CNOT decomposition reversed)
show(
    "CNOT(0,1) · CNOT(1,0) · CNOT(0,1)  →  SWAP(0,1)",
    [(CNOT_gate, (0,1), False), (CNOT_gate, (1,0), False), (CNOT_gate, (0,1), False)],
    n=2, expect_len=1,
)

[CNOT(0,1) · CNOT(1,0) · CNOT(0,1)  →  SWAP(0,1)]
  before ( 3): CNOT(0,1) · CNOT(1,0) · CNOT(0,1)
  after  ( 1): SWAP(0,1)  ✓



In [13]:
# H(a) · H(b) · CNOT(a,b) · H(a) · H(b) = CNOT(b,a)  (Hadamard conjugation flips direction)
show(
    "H(0)·H(1) · CNOT(0,1) · H(0)·H(1)  →  CNOT(1,0)",
    [
        (H_gate, (0,), False), (H_gate, (1,), False),
        (CNOT_gate, (0,1), False),
        (H_gate, (0,), False), (H_gate, (1,), False),
    ],
    n=2, expect_len=1,
)

[H(0)·H(1) · CNOT(0,1) · H(0)·H(1)  →  CNOT(1,0)]
  before ( 5): H(0) · H(1) · CNOT(0,1) · H(0) · H(1)
  after  ( 1): CNOT(1,0)  ✓



In [14]:
# Control check: H on only ONE side does NOT reduce
from qudit_sim.circuit import QuantumCircuit as QC

ops_one_side = [(H_gate, (0,), False), (CNOT_gate, (0,1), False), (H_gate, (0,), False)]
opt_one_side = optimize_ops(list(ops_one_side))
U_before = get_unitary(ops_one_side, 2)
U_flipped = get_unitary([(CNOT_gate, (1,0), False)], 2)
print(f"H(0)·CNOT(0,1)·H(0): {len(ops_one_side)} → {len(opt_one_side)} gates")
print(f"  equals CNOT(1,0)? {np.allclose(U_before, U_flipped)}  (should be False — this rule does NOT hold)")

H(0)·CNOT(0,1)·H(0): 3 → 3 gates
  equals CNOT(1,0)? False  (should be False — this rule does NOT hold)


---
## 4. Fixed-Point Iteration

Both passes run in a loop until the circuit length stops decreasing.
This catches chains where one reduction exposes another.

In [15]:
# H·Z·H produces X; the trailing X then cancels with it → empty circuit (2 iterations)
show(
    "H · Z · H · X  →  ∅  (H·Z·H→X, then X·X→I)",
    [(H_gate,(0,),False),(Z_gate,(0,),False),(H_gate,(0,),False),(X_gate,(0,),False)],
    n=1, expect_len=0,
)

[H · Z · H · X  →  ∅  (H·Z·H→X, then X·X→I)]
  before ( 4): H(0) · Z(0) · H(0) · X(0)
  after  ( 0): ∅  (identity)  ✓



In [16]:
# Four H gates cancel in two rounds
show(
    "H · H · H · H  →  ∅",
    [(H_gate,(0,),False)]*4,
    n=1, expect_len=0,
)

[H · H · H · H  →  ∅]
  before ( 4): H(0) · H(0) · H(0) · H(0)
  after  ( 0): ∅  (identity)  ✓



In [17]:
# Combined: CNOT decomposition + commutation-cancel
# SWAP is produced by CNOT·CNOT·CNOT; a second SWAP cancels it → identity
show(
    "CNOT(0,1)·CNOT(1,0)·CNOT(0,1) · SWAP(0,1)  →  ∅  (3-CNOT→SWAP, then SWAP·SWAP→I)",
    [
        (CNOT_gate,(0,1),False),(CNOT_gate,(1,0),False),(CNOT_gate,(0,1),False),
        (SWAP_gate,(0,1),False),
    ],
    n=2, expect_len=0,
)

[CNOT(0,1)·CNOT(1,0)·CNOT(0,1) · SWAP(0,1)  →  ∅  (3-CNOT→SWAP, then SWAP·SWAP→I)]
  before ( 4): CNOT(0,1) · CNOT(1,0) · CNOT(0,1) · SWAP(0,1)
  after  ( 0): ∅  (identity)  ✓



---
## 5. Stress Test — Unitary Preservation on Random Circuits

The optimizer must never change the circuit's unitary, even on circuits where nothing cancels.
We sample 500 random circuits (1–3 qubits, 0–15 gates) and verify this property.

In [18]:
GATE_POOL_1Q = [H_gate, X_gate, Z_gate]
GATE_POOL_2Q = [CNOT_gate, SWAP_gate]

rng = np.random.default_rng(0)
n_tests, failures, total_before, total_after = 500, 0, 0, 0

for _ in range(n_tests):
    n = int(rng.integers(1, 4))
    ops = []
    for _ in range(int(rng.integers(0, 16))):
        if n == 1 or rng.random() < 0.5:
            g = GATE_POOL_1Q[rng.integers(len(GATE_POOL_1Q))]
            q = (int(rng.integers(n)),)
        else:
            g = GATE_POOL_2Q[rng.integers(len(GATE_POOL_2Q))]
            q = tuple(int(x) for x in rng.choice(n, 2, replace=False))
        ops.append((g, q, False))

    opt = optimize_ops(list(ops))
    U_b = get_unitary(ops, n)
    U_a = get_unitary(opt, n)
    if not np.allclose(U_b, U_a, atol=1e-10):
        failures += 1
    total_before += len(ops)
    total_after  += len(opt)

reduction = 100 * (1 - total_after / total_before) if total_before else 0
print(f"Tests run   : {n_tests}")
print(f"Failures    : {failures}")
print(f"Total gates : {total_before} → {total_after}  ({reduction:.1f}% reduction)")

Tests run   : 500
Failures    : 0
Total gates : 3710 → 2298  (38.1% reduction)
